# Empirical Resource Tracking

> Persistent store for empirically-observed resource usage per (instance_id, config_hash) pair. CR-7's data foundation — `record_sample` is called from `CapabilityManager.execute_capability*` finally blocks; aggregates feed eviction-candidate selection + future UI hints + cost-aware retry decisions.

In [ ]:
#| default_exp core.empirical_store

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
import sqlite3
from contextlib import contextmanager
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterator, List, Optional, Protocol, runtime_checkable

import logging
_logger = logging.getLogger(__name__)

from cjm_substrate.utils.hashing import hash_dict_canonical

## `compute_config_hash`

Canonical hash of a capability instance's effective config — the second component of the (instance_id, config_hash) compound key. Two distinct configs for the same instance get two distinct empirical records, because their resource profiles differ (e.g. Whisper at model='base' vs Whisper at model='large-v3'). Reuses `hash_dict_canonical` from `cjm_substrate.utils.hashing`, the same primitive CR-8 uses for `compute_config_schema_hash` — so stability rules are uniform across the substrate.

In [ ]:
#| export
def compute_config_hash(
    config: Optional[Dict[str, Any]],  # Capability's effective config dict (or None / empty)
) -> str:  # Hash string in "algo:hexdigest" format
    """CR-7: hash a capability instance's effective config for empirical-record keying.
    
    Same canonicalization as CR-8's `compute_config_schema_hash` — sorted keys,
    no whitespace, `"sha256:hex"` shape. None / empty configs hash deterministically
    to the canonical-empty value so capabilities with no config still get a single
    record per instance rather than scattering across hash-of-None edge cases.
    """
    return hash_dict_canonical(config)

## `ResourceSample`

One observation captured at the end of an execute call. Frozen because it's an immutable record of what happened. Substrate aggregates samples online into a running `EmpiricalResourceRecord` and discards each sample after — no per-sample retention in v1.

**Important v1 limitation**: `cpu_percent` / `memory_mb_peak` / `gpu_memory_mb_peak` come from `proxy.get_stats()` at the moment we measure, not from continuous sampling during execute. They're a proxy for peak ("what was the worker using when execute returned"), not the true peak ("what was the worker using at any moment during execute"). Worker-side peak tracking is a future enhancement when needed.

In [ ]:
#| export
@dataclass(frozen=True)
class ResourceSample:
    """Single observation captured after an execute call completes.
    
    Frozen — substrate aggregates online via Welford's algorithm; no need to
    keep raw samples around. `observed_at` is tz-aware per the CR-5 convention.
    """
    cpu_percent: float  # Worker CPU% at end-of-execute (proxy for peak)
    memory_mb_peak: float  # Worker RSS in MB at end-of-execute (proxy for peak)
    gpu_memory_mb_peak: float  # Worker GPU memory in MB at end-of-execute (0.0 if no GPU)
    duration_seconds: float  # Wall-clock execute duration
    success: bool  # True if execute returned normally; False if raised
    observed_at: datetime  # tz-aware datetime — when the sample was captured
    api_usage: Optional[Dict[str, float]] = None  # SG-54: unit-agnostic measured usage {unit_name: amount}; None for compute-only capabilities

## `EmpiricalResourceRecord`

Aggregated profile for a (instance_id, config_hash) pair. Welford-mean for CPU + duration; max-of-peaks AND running mean for memory metrics (the worst observation drives eviction-candidate selection; the mean smooths over noise for UI hints / capacity-planning).

`success_rate = success_count / sample_count` — exposed pre-computed; substrate doesn't surface raw `success_count` / `failure_count`.

In [ ]:
#| export
@dataclass
class EmpiricalResourceRecord:
    """Aggregated empirical resource profile for a (instance_id, config_hash) pair."""
    instance_id: str  # CapabilityInstance.instance_id (CR-10 multi-instance aware)
    capability_name: str  # Convenience: CapabilityInstance.capability_name; derivable but cheap to denormalize
    config_hash: str  # compute_config_hash(inst.config) at sample time
    sample_count: int  # Number of ResourceSamples folded into this record
    cpu_percent_mean: float  # Welford running mean of cpu_percent
    memory_mb_peak_max: float  # max(sample.memory_mb_peak over all samples) — worst observed
    memory_mb_peak_mean: float  # Welford running mean of sample.memory_mb_peak
    gpu_memory_mb_peak_max: float  # max(sample.gpu_memory_mb_peak over all samples)
    gpu_memory_mb_peak_mean: float  # Welford running mean of sample.gpu_memory_mb_peak
    duration_seconds_mean: float  # Welford running mean of sample.duration_seconds
    success_rate: float  # success_count / sample_count
    last_observed: datetime  # tz-aware; tracks most recent ResourceSample.observed_at
    api_usage_totals: Dict[str, float] = field(default_factory=dict)  # SG-54: cumulative per-unit usage summed across runs (tokens/credits/pages/...); {} for compute-only capabilities

## `EmpiricalResourceStore` Protocol

Protocol for persisting empirical resource records. Mirrors `CapabilityConfigStore` from CR-2 / OQ-4 — substrate ships `LocalEmpiricalResourceStore` (SQLite) as the default; hosts pass an explicit implementation to constrain to in-memory (tests), workflow-scoped storage, in-process Redis, etc.

`@runtime_checkable` so `isinstance(x, EmpiricalResourceStore)` works for substrate code that wants to assert the seam without importing the concrete class.

In [ ]:
#| export
@runtime_checkable
class EmpiricalResourceStore(Protocol):
    """Protocol for persisting empirically-observed resource usage.
    
    Implementations aggregate online (Welford for means, max-of-peaks for memory).
    No raw-sample retention required — v1 is one row per (instance_id, config_hash)
    pair with running aggregates. A future implementation can add a samples table
    if time-series queries become necessary.
    """
    
    def record_sample(
        self,
        instance_id: str,
        capability_name: str,
        config_hash: str,
        sample: ResourceSample,
    ) -> None:
        """Fold a sample into the running aggregate. Creates a new record on first call."""
        ...
    
    def get_record(
        self,
        instance_id: str,
        config_hash: str,
    ) -> Optional[EmpiricalResourceRecord]:
        """Fetch the aggregated record for (instance_id, config_hash), or None."""
        ...
    
    def list_records(
        self,
        capability_name: Optional[str] = None,
    ) -> List[EmpiricalResourceRecord]:
        """List all records, optionally filtered to a single capability_name."""
        ...
    
    def delete_record(
        self,
        instance_id: str,
        config_hash: str,
    ) -> bool:
        """Remove a record. Returns True if a row was deleted."""
        ...

## `LocalEmpiricalResourceStore`

SQLite-backed default. Mirrors `LocalCapabilityConfigStore` — DB created lazily on first write, reads against a non-existent DB return None / empty rather than raising. Default path is `~/.cjm/empirical_resources.db` (override via `db_path=...` in the constructor or via per-project `cjm.yaml` data_dir).

**Welford's algorithm**: for each new sample x, update running mean via `mean += (x - mean) / n` where `n` is the new sample count. Trivial cost; numerically stable across long sample runs. We don't track the variance accumulator M2 in v1 since no consumer exposes variance — adding it later is a schema-additive change.

In [ ]:
#| export
from fastcore.basics import patch


In [ ]:
#| export
_SCHEMA = """
CREATE TABLE IF NOT EXISTS empirical_resources (
    instance_id TEXT NOT NULL,
    capability_name TEXT NOT NULL,
    config_hash TEXT NOT NULL,
    sample_count INTEGER NOT NULL DEFAULT 0,
    success_count INTEGER NOT NULL DEFAULT 0,
    cpu_percent_mean REAL NOT NULL DEFAULT 0.0,
    memory_mb_peak_max REAL NOT NULL DEFAULT 0.0,
    memory_mb_peak_mean REAL NOT NULL DEFAULT 0.0,
    gpu_memory_mb_peak_max REAL NOT NULL DEFAULT 0.0,
    gpu_memory_mb_peak_mean REAL NOT NULL DEFAULT 0.0,
    duration_seconds_mean REAL NOT NULL DEFAULT 0.0,
    last_observed TEXT NOT NULL,
    api_usage_totals TEXT NOT NULL DEFAULT '{}',
    PRIMARY KEY (instance_id, config_hash)
)
"""


def _default_db_path() -> Path:
    """Default SQLite location: `~/.cjm/empirical_resources.db`.
    
    Hosts using per-project `data_dir` (the intended pattern per CR-8 cascade_manifests
    docs) override this by passing `db_path=cfg.data_dir / "empirical_resources.db"`
    when constructing the store. CapabilityManager's lazy-init does this automatically.
    """
    return Path.home() / ".cjm" / "empirical_resources.db"


class LocalEmpiricalResourceStore:
    """SQLite-backed default implementation of `EmpiricalResourceStore`.
    
    Online Welford aggregation for means; max-of-peaks for memory metrics.
    success_rate computed at read time from `success_count / sample_count`.
    DB + schema created lazily on first write.
    """
    
    def __init__(self, db_path: Optional[Path] = None):
        """Initialize the store. `db_path=None` uses `~/.cjm/empirical_resources.db`."""
        self.db_path = Path(db_path) if db_path is not None else _default_db_path()
    
    
    
    
    
    
    @staticmethod
    def _row_to_record(row) -> EmpiricalResourceRecord:
        """Convert a SQLite row tuple to a typed EmpiricalResourceRecord.
        
        success_rate computed at read time. last_observed parsed back to tz-aware
        datetime (substrate-side records always serialize via isoformat()).
        """
        (instance_id, capability_name, config_hash, sample_count, success_count,
         cpu_mean, mem_max, mem_mean, gpu_max, gpu_mean,
         dur_mean, last_observed_str, api_usage_totals_json) = row
        try:
            api_usage_totals = json.loads(api_usage_totals_json) if api_usage_totals_json else {}
        except (TypeError, ValueError):
            api_usage_totals = {}
        success_rate = (success_count / sample_count) if sample_count > 0 else 0.0
        try:
            last_observed = datetime.fromisoformat(last_observed_str)
        except (TypeError, ValueError):
            last_observed = datetime.now(timezone.utc)
        return EmpiricalResourceRecord(
            instance_id=instance_id,
            capability_name=capability_name,
            config_hash=config_hash,
            sample_count=int(sample_count),
            cpu_percent_mean=float(cpu_mean),
            memory_mb_peak_max=float(mem_max),
            memory_mb_peak_mean=float(mem_mean),
            gpu_memory_mb_peak_max=float(gpu_max),
            gpu_memory_mb_peak_mean=float(gpu_mean),
            duration_seconds_mean=float(dur_mean),
            success_rate=success_rate,
            last_observed=last_observed,
            api_usage_totals=api_usage_totals,
        )

In [ ]:
#| export
@patch
@contextmanager
def _conn(self:LocalEmpiricalResourceStore) -> Iterator[sqlite3.Connection]:
    """Open a connection, creating parent dirs + schema on demand."""
    self.db_path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(self.db_path)
    try:
        conn.execute(_SCHEMA)
        # SG-54: in-place migration for DBs created before api_usage_totals.
        cols = {r[1] for r in conn.execute("PRAGMA table_info(empirical_resources)").fetchall()}
        if "api_usage_totals" not in cols:
            conn.execute("ALTER TABLE empirical_resources ADD COLUMN api_usage_totals TEXT NOT NULL DEFAULT '{}'")
        conn.commit()
        yield conn
    finally:
        conn.close()

In [ ]:
#| export
@patch
def record_sample(
    self:LocalEmpiricalResourceStore,
    instance_id: str,  # CapabilityInstance.instance_id
    capability_name: str,  # CapabilityInstance.capability_name (denormalized for filtering)
    config_hash: str,  # compute_config_hash(inst.config)
    sample: ResourceSample,  # One observation
) -> None:
    """Fold a sample into the running aggregate. Creates a new row on first call.

    Welford update for each mean. Max-of-peaks for memory metrics.
    success_count incremented by 1 if sample.success else 0.
    """
    with self._conn() as conn:
        row = conn.execute(
            "SELECT sample_count, success_count, cpu_percent_mean, "
            "memory_mb_peak_max, memory_mb_peak_mean, "
            "gpu_memory_mb_peak_max, gpu_memory_mb_peak_mean, "
            "duration_seconds_mean, api_usage_totals "
            "FROM empirical_resources WHERE instance_id = ? AND config_hash = ?",
            (instance_id, config_hash),
        ).fetchone()

        success_delta = 1 if sample.success else 0
        if row is None:
            # First sample for this (instance_id, config_hash) pair.
            n = 1
            success_count = success_delta
            cpu_mean = sample.cpu_percent
            mem_mean = sample.memory_mb_peak
            gpu_mean = sample.gpu_memory_mb_peak
            dur_mean = sample.duration_seconds
            mem_max = sample.memory_mb_peak
            gpu_max = sample.gpu_memory_mb_peak
            usage_totals = {k: float(v) for k, v in (sample.api_usage or {}).items()}
        else:
            (old_n, old_success_count, old_cpu_mean,
             old_mem_max, old_mem_mean,
             old_gpu_max, old_gpu_mean,
             old_dur_mean, old_usage_totals_json) = row
            n = old_n + 1
            success_count = old_success_count + success_delta
            # Welford running-mean update: mean_n = mean_{n-1} + (x - mean_{n-1}) / n
            cpu_mean = old_cpu_mean + (sample.cpu_percent - old_cpu_mean) / n
            mem_mean = old_mem_mean + (sample.memory_mb_peak - old_mem_mean) / n
            gpu_mean = old_gpu_mean + (sample.gpu_memory_mb_peak - old_gpu_mean) / n
            dur_mean = old_dur_mean + (sample.duration_seconds - old_dur_mean) / n
            # Max-of-peaks for memory metrics (eviction-candidate selection reads this)
            mem_max = max(old_mem_max, sample.memory_mb_peak)
            gpu_max = max(old_gpu_max, sample.gpu_memory_mb_peak)
            # SG-54: usage is ADDITIVE (cumulative) — merge-sum per unit name.
            try:
                usage_totals = json.loads(old_usage_totals_json) if old_usage_totals_json else {}
            except (TypeError, ValueError):
                usage_totals = {}
            for _k, _v in (sample.api_usage or {}).items():
                usage_totals[_k] = float(usage_totals.get(_k, 0.0)) + float(_v)

        conn.execute(
            "INSERT OR REPLACE INTO empirical_resources ("
            "instance_id, capability_name, config_hash, sample_count, success_count, "
            "cpu_percent_mean, memory_mb_peak_max, memory_mb_peak_mean, "
            "gpu_memory_mb_peak_max, gpu_memory_mb_peak_mean, "
            "duration_seconds_mean, last_observed, api_usage_totals"
            ") VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
            (
                instance_id, capability_name, config_hash, n, success_count,
                cpu_mean, mem_max, mem_mean, gpu_max, gpu_mean,
                dur_mean, sample.observed_at.isoformat(), json.dumps(usage_totals, sort_keys=True),
            ),
        )
        conn.commit()

In [ ]:
#| export
@patch
def get_record(
    self:LocalEmpiricalResourceStore,
    instance_id: str,
    config_hash: str,
) -> Optional[EmpiricalResourceRecord]:
    """Fetch the aggregated record for (instance_id, config_hash), or None."""
    if not self.db_path.exists():
        return None
    with self._conn() as conn:
        row = conn.execute(
            "SELECT instance_id, capability_name, config_hash, sample_count, success_count, "
            "cpu_percent_mean, memory_mb_peak_max, memory_mb_peak_mean, "
            "gpu_memory_mb_peak_max, gpu_memory_mb_peak_mean, "
            "duration_seconds_mean, last_observed, api_usage_totals "
            "FROM empirical_resources WHERE instance_id = ? AND config_hash = ?",
            (instance_id, config_hash),
        ).fetchone()
    return self._row_to_record(row) if row is not None else None

In [ ]:
#| export
@patch
def list_records(
    self:LocalEmpiricalResourceStore,
    capability_name: Optional[str] = None,
) -> List[EmpiricalResourceRecord]:
    """List all records, optionally filtered to a capability."""
    if not self.db_path.exists():
        return []
    with self._conn() as conn:
        if capability_name is not None:
            rows = conn.execute(
                "SELECT instance_id, capability_name, config_hash, sample_count, success_count, "
                "cpu_percent_mean, memory_mb_peak_max, memory_mb_peak_mean, "
                "gpu_memory_mb_peak_max, gpu_memory_mb_peak_mean, "
                "duration_seconds_mean, last_observed, api_usage_totals "
                "FROM empirical_resources WHERE capability_name = ?",
                (capability_name,),
            ).fetchall()
        else:
            rows = conn.execute(
                "SELECT instance_id, capability_name, config_hash, sample_count, success_count, "
                "cpu_percent_mean, memory_mb_peak_max, memory_mb_peak_mean, "
                "gpu_memory_mb_peak_max, gpu_memory_mb_peak_mean, "
                "duration_seconds_mean, last_observed, api_usage_totals "
                "FROM empirical_resources",
            ).fetchall()
    return [self._row_to_record(r) for r in rows]

In [ ]:
#| export
@patch
def delete_record(
    self:LocalEmpiricalResourceStore,
    instance_id: str,
    config_hash: str,
) -> bool:
    """Remove a record. Returns True if a row was deleted."""
    if not self.db_path.exists():
        return False
    with self._conn() as conn:
        cur = conn.execute(
            "DELETE FROM empirical_resources WHERE instance_id = ? AND config_hash = ?",
            (instance_id, config_hash),
        )
        conn.commit()
        return cur.rowcount > 0

## Tests

In [ ]:
#| hide
# compute_config_hash: insertion-order independent, None == {}, sha256-tagged.
h1 = compute_config_hash({"model": "base", "device": "cuda"})
h2 = compute_config_hash({"device": "cuda", "model": "base"})
assert h1 == h2, "insertion-order must not affect the hash"

h_none = compute_config_hash(None)
h_empty = compute_config_hash({})
assert h_none == h_empty, "None and {} hash identically"

assert h1.startswith("sha256:"), f"expected algo-tagged hash, got {h1!r}"

h_diff = compute_config_hash({"model": "large"})
assert h1 != h_diff, "different configs must produce different hashes"

print("✓ compute_config_hash: order-independent, canonical-empty, sha256-tagged, sensitive")

In [ ]:
#| hide
# Welford aggregation: incremental mean tracks true mean; max-of-peaks is monotonic;
# success_rate computes correctly from success_count / sample_count.
import tempfile
import os as _os

fd, db_path = tempfile.mkstemp(suffix=".db")
_os.close(fd)
_os.unlink(db_path)

try:
    store = LocalEmpiricalResourceStore(Path(db_path))
    
    # No record yet
    assert store.get_record("whisper", "abc") is None
    assert store.list_records() == []
    
    # Record three samples with known values; verify Welford mean matches
    # the arithmetic mean computed independently.
    samples = [
        ResourceSample(cpu_percent=20.0, memory_mb_peak=1000.0, gpu_memory_mb_peak=5000.0,
                       duration_seconds=10.0, success=True,
                       observed_at=datetime.now(timezone.utc)),
        ResourceSample(cpu_percent=40.0, memory_mb_peak=1500.0, gpu_memory_mb_peak=7000.0,
                       duration_seconds=20.0, success=True,
                       observed_at=datetime.now(timezone.utc)),
        ResourceSample(cpu_percent=60.0, memory_mb_peak=1200.0, gpu_memory_mb_peak=6500.0,
                       duration_seconds=15.0, success=False,
                       observed_at=datetime.now(timezone.utc)),
    ]
    for s in samples:
        store.record_sample("whisper-base", "whisper", "sha256:abc", s)
    
    rec = store.get_record("whisper-base", "sha256:abc")
    assert rec is not None
    assert rec.sample_count == 3
    # Arithmetic means computed independently — should match Welford to floating-point precision
    expected_cpu_mean = (20.0 + 40.0 + 60.0) / 3
    expected_mem_mean = (1000.0 + 1500.0 + 1200.0) / 3
    expected_gpu_mean = (5000.0 + 7000.0 + 6500.0) / 3
    expected_dur_mean = (10.0 + 20.0 + 15.0) / 3
    assert abs(rec.cpu_percent_mean - expected_cpu_mean) < 1e-9
    assert abs(rec.memory_mb_peak_mean - expected_mem_mean) < 1e-9
    assert abs(rec.gpu_memory_mb_peak_mean - expected_gpu_mean) < 1e-9
    assert abs(rec.duration_seconds_mean - expected_dur_mean) < 1e-9
    
    # Max-of-peaks for memory metrics
    assert rec.memory_mb_peak_max == 1500.0, "max-of-peaks must track the worst observation"
    assert rec.gpu_memory_mb_peak_max == 7000.0
    
    # 2 of 3 samples succeeded
    assert abs(rec.success_rate - 2/3) < 1e-9
    
    # last_observed survived round-trip
    assert rec.last_observed.tzinfo is not None, "last_observed must be tz-aware"
    
    print("✓ Welford mean + max-of-peaks + success_rate aggregation correct")
finally:
    try:
        _os.unlink(db_path)
    except OSError:
        pass

In [ ]:
#| hide
# Different (instance_id, config_hash) pairs get distinct records. Same instance
# with different configs gets two records. Same config but different instances
# (CR-10 multi-instance) also gets two records.
import tempfile
import os as _os

fd, db_path = tempfile.mkstemp(suffix=".db")
_os.close(fd)
_os.unlink(db_path)

try:
    store = LocalEmpiricalResourceStore(Path(db_path))
    
    s1 = ResourceSample(cpu_percent=10.0, memory_mb_peak=100.0, gpu_memory_mb_peak=0.0,
                        duration_seconds=1.0, success=True,
                        observed_at=datetime.now(timezone.utc))
    s2 = ResourceSample(cpu_percent=50.0, memory_mb_peak=500.0, gpu_memory_mb_peak=0.0,
                        duration_seconds=5.0, success=True,
                        observed_at=datetime.now(timezone.utc))
    
    # Two different config hashes for the same instance — separate records
    store.record_sample("whisper", "whisper", compute_config_hash({"model": "base"}), s1)
    store.record_sample("whisper", "whisper", compute_config_hash({"model": "large"}), s2)
    
    base_rec = store.get_record("whisper", compute_config_hash({"model": "base"}))
    large_rec = store.get_record("whisper", compute_config_hash({"model": "large"}))
    assert base_rec is not None and large_rec is not None
    assert base_rec.cpu_percent_mean == 10.0
    assert large_rec.cpu_percent_mean == 50.0
    
    # Two different instances (CR-10 multi-instance) for the same capability — separate records
    cfg_hash = compute_config_hash({"model": "base"})
    store.record_sample("whisper-a", "whisper", cfg_hash, s1)
    store.record_sample("whisper-b", "whisper", cfg_hash, s2)
    
    a_rec = store.get_record("whisper-a", cfg_hash)
    b_rec = store.get_record("whisper-b", cfg_hash)
    assert a_rec is not None and b_rec is not None
    assert a_rec.cpu_percent_mean == 10.0
    assert b_rec.cpu_percent_mean == 50.0
    
    # list_records: all records
    all_recs = store.list_records()
    assert len(all_recs) == 4, f"expected 4 records, got {len(all_recs)}"
    
    # list_records filtered by capability_name
    whisper_recs = store.list_records(capability_name="whisper")
    assert len(whisper_recs) == 4, "all four are 'whisper' capability"
    none_recs = store.list_records(capability_name="nonexistent")
    assert none_recs == []
    
    print("✓ Multi-instance + multi-config keying: distinct records per (instance_id, config_hash)")
finally:
    try:
        _os.unlink(db_path)
    except OSError:
        pass

In [ ]:
#| hide
# delete_record returns True/False correctly; reads from a non-existent DB
# return None / empty rather than raising.
import tempfile
import os as _os

fd, db_path = tempfile.mkstemp(suffix=".db")
_os.close(fd)
_os.unlink(db_path)

# Non-existent DB: get/list return None/empty; delete returns False (no row)
store = LocalEmpiricalResourceStore(Path(db_path))
assert store.get_record("x", "y") is None
assert store.list_records() == []
assert store.delete_record("x", "y") is False

# Now add a record and delete it
s = ResourceSample(cpu_percent=1.0, memory_mb_peak=1.0, gpu_memory_mb_peak=0.0,
                   duration_seconds=1.0, success=True,
                   observed_at=datetime.now(timezone.utc))
store.record_sample("a", "a", "sha256:xyz", s)
assert store.get_record("a", "sha256:xyz") is not None

assert store.delete_record("a", "sha256:xyz") is True, "delete must return True on hit"
assert store.get_record("a", "sha256:xyz") is None
assert store.delete_record("a", "sha256:xyz") is False, "second delete is a miss"

try:
    _os.unlink(db_path)
except OSError:
    pass

# Protocol type-check works (runtime_checkable)
assert isinstance(store, EmpiricalResourceStore), \
    "LocalEmpiricalResourceStore must satisfy EmpiricalResourceStore Protocol"

print("✓ delete_record + empty-DB safety + Protocol type-check")

In [ ]:
#| hide
# SG-54: api_usage_totals accumulates per-unit SUMS (additive + unit-agnostic);
# absent usage leaves {}; a pre-SG-54 DB upgrades in place via ALTER.
import tempfile, sqlite3 as _sql
from pathlib import Path as _P

def _mk(usage=None, success=True):
    return ResourceSample(0.0, 0.0, 0.0, 0.1, success, datetime.now(timezone.utc), api_usage=usage)

def _sg54_usage_accumulation():
    with tempfile.TemporaryDirectory() as d:
        store = LocalEmpiricalResourceStore(_P(d) / "e.db")
        store.record_sample("gem", "gemini", "h1", _mk({"input_tokens": 10, "output_tokens": 5, "total_tokens": 15}))
        store.record_sample("gem", "gemini", "h1", _mk({"input_tokens": 2, "output_tokens": 3, "total_tokens": 5}))
        rec = store.get_record("gem", "h1")
        assert rec.api_usage_totals == {"input_tokens": 12.0, "output_tokens": 8.0, "total_tokens": 20.0}, rec.api_usage_totals
        assert rec.sample_count == 2
        # A run reporting a NEW unit key accumulates independently.
        store.record_sample("gem", "gemini", "h1", _mk({"requests": 1, "input_tokens": 1}))
        rec = store.get_record("gem", "h1")
        assert rec.api_usage_totals["requests"] == 1.0 and rec.api_usage_totals["input_tokens"] == 13.0
        # Compute-only capability (no usage reported) -> empty totals (cost-axis stays blind, correctly).
        store.record_sample("cpu", "nltk", "h2", _mk(None))
        assert store.get_record("cpu", "h2").api_usage_totals == {}

def _sg54_migration():
    with tempfile.TemporaryDirectory() as d:
        dbp = _P(d) / "old.db"
        conn = _sql.connect(dbp)
        conn.execute(
            "CREATE TABLE empirical_resources ("
            "instance_id TEXT NOT NULL, capability_name TEXT NOT NULL, config_hash TEXT NOT NULL, "
            "sample_count INTEGER NOT NULL DEFAULT 0, success_count INTEGER NOT NULL DEFAULT 0, "
            "cpu_percent_mean REAL NOT NULL DEFAULT 0.0, memory_mb_peak_max REAL NOT NULL DEFAULT 0.0, "
            "memory_mb_peak_mean REAL NOT NULL DEFAULT 0.0, gpu_memory_mb_peak_max REAL NOT NULL DEFAULT 0.0, "
            "gpu_memory_mb_peak_mean REAL NOT NULL DEFAULT 0.0, duration_seconds_mean REAL NOT NULL DEFAULT 0.0, "
            "last_observed TEXT NOT NULL, PRIMARY KEY (instance_id, config_hash))"
        )
        conn.execute(
            "INSERT INTO empirical_resources VALUES ('old','p','h',1,1,0,0,0,0,0,0.1,?)",
            (datetime.now(timezone.utc).isoformat(),),
        )
        conn.commit(); conn.close()
        # Opening with the new store ALTERs the column in; old row reads as {}.
        store = LocalEmpiricalResourceStore(dbp)
        rec = store.get_record("old", "h")
        assert rec is not None and rec.api_usage_totals == {}
        store.record_sample("old", "p", "h", _mk({"total_tokens": 7}))
        assert store.get_record("old", "h").api_usage_totals == {"total_tokens": 7.0}

_sg54_usage_accumulation()
_sg54_migration()
print("SG-54 api_usage_totals accumulation + in-place migration: PASS")


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()